# EMPIEZA ADOLFO

# Learning Urban Crimes Representation

## Aprendizaje de representaciones urbanas
Input:  firmas_h3.csv (1,061 hexágonos × 45 dims)
        h3_metadata.csv (metadatos por hexágono)

Métodos:
  A) Línea base — PCA, NMF, K-Means, GMM, HDBSCAN
  B) Autoencoder denso
  C) Graph Autoencoder (GAE) — usa vecindad H3
  D) Variational Graph Autoencoder (VGAE) — versión probabilística
  E) Comparación de todas las representaciones

Output: embeddings_h3.csv — embeddings de todos los métodos
        clusters_h3.csv — clusters por método
        modelo_autoencoder.pth — pesos autoencoder denso
        modelo_gae.pth — pesos GAE
        modelo_vgae.pth — pesos VGAE

### Paquetes

In [46]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA, NMF
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, calinski_harabasz_score, adjusted_rand_score, normalized_mutual_info_score, roc_auc_score, average_precision_score
import hdbscan

# autoencodes
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# GAE
from torch_geometric.nn import GCNConv
from torch_geometric.utils import negative_sampling

import itertools

import warnings
warnings.filterwarnings('ignore')

import h3

### Ejecución

#### Configuración de torch

In [36]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f"Device: {device}")

Device: cuda


#### Cargar datos

In [3]:
# cargar firmas y metadata de los hexágonos
firmas = pd.read_csv("../data/auxiliar/firmas_h3.csv", index_col='h3_id')
metadata = pd.read_csv("../data/auxiliar/h3_metadata.csv", index_col='h3_id')

print(f"Hexágonos: {len(firmas):,}")
print(f"Dimensiones: {firmas.shape[1]}")

Hexágonos: 1,061
Dimensiones: 45


#### Implementación

In [4]:
# ============================================================================
# PASO 1: Preprocesamiento
# ============================================================================
# Separar features proporcionales de features absolutas
#cols_proporciones = [c for c in firmas.columns if c not in ['intensidad_log', 'ratio_violencia']]
#cols_extra = ['intensidad_log', 'ratio_violencia']
# TODO esto de cols no se usa

# Para PCA y Autoencoder: estandarizar todo
scaler_std = StandardScaler()
X_std = scaler_std.fit_transform(firmas)

# Para NMF: usar MinMax (NMF requiere valores no negativos)
scaler_mm = MinMaxScaler()
X_mm = scaler_mm.fit_transform(firmas)

feature_names = firmas.columns.tolist()
h3_ids = firmas.index.tolist()

print(f"Datos estandarizados: {X_std.shape}")

Datos estandarizados: (1061, 45)


In [5]:
# ============================================================================
# PASO 2A: PCA
# ============================================================================
print(f"\n{'='*80}")
print(f"Método A1: PCA")
print(f"{'='*80}")

# PCA completo para ver varianza explicada
pca_full = PCA().fit(X_std)
varianza_acum = np.cumsum(pca_full.explained_variance_ratio_)

# ¿Cuántas componentes para 80%, 90%, 95%?
for umbral in [0.80, 0.90, 0.95]:
    n_comp = np.argmax(varianza_acum >= umbral) + 1
    print(f"  Componentes para {umbral*100:.0f}% varianza: {n_comp}")

# Usar componentes que expliquen 90% de la varianza
N_COMPONENTS_PCA = np.argmax(varianza_acum >= 0.90) + 1
pca = PCA(n_components=N_COMPONENTS_PCA)
X_pca = pca.fit_transform(X_std) # Reducción PCA (resultados)

print(f"\n  PCA con {N_COMPONENTS_PCA} componentes:")
print(f"  Varianza explicada: {pca.explained_variance_ratio_.sum()*100:.1f}%")

# Top features por componente (primeras 3)
print(f"\n  Top features por componente:")
for i in range(min(3, N_COMPONENTS_PCA)):
    loadings = pd.Series(pca.components_[i], index=feature_names)
    top_pos = loadings.nlargest(3)
    top_neg = loadings.nsmallest(3)
    print(f"\n  PC{i+1} ({pca.explained_variance_ratio_[i]*100:.1f}% varianza):")
    print(f"    (+) {', '.join([f'{n}: {v:.3f}' for n, v in top_pos.items()])}")
    print(f"    (-) {', '.join([f'{n}: {v:.3f}' for n, v in top_neg.items()])}")


Método A1: PCA
  Componentes para 80% varianza: 22
  Componentes para 90% varianza: 29
  Componentes para 95% varianza: 34

  PCA con 29 componentes:
  Varianza explicada: 90.7%

  Top features por componente:

  PC1 (16.5% varianza):
    (+) delito_violencia_familiar: 0.309, dia_sunday: 0.264, hora_noche: 0.246
    (-) delito_falsificacion_y_documentos: -0.284, delito_robo_sin_violencia: -0.281, delito_fraude_y_delitos_patrimoniales: -0.217

  PC2 (8.1% varianza):
    (+) delito_robo_con_violencia: 0.423, ratio_violencia: 0.385, intensidad_log: 0.248
    (-) hora_manana: -0.346, delito_fraude_y_delitos_patrimoniales: -0.289, delito_delitos_ambientales: -0.244

  PC3 (5.3% varianza):
    (+) delito_lesiones_culposas: 0.345, delito_homicidio: 0.310, delito_dano_en_propiedad: 0.256
    (-) delito_delitos_de_servidores_publicos: -0.276, dia_monday: -0.225, delito_amenazas: -0.214


In [6]:
# ============================================================================
# PASO 2B: NMF
# ============================================================================
print(f"\n{'='*80}")
print(f"Método A2: Non-Negative Matrix Factorization (NMF)")
print(f"{'='*80}")

# NMF con el mismo número de componentes
nmf = NMF(n_components=N_COMPONENTS_PCA, init='nndsvda', max_iter=500, random_state=42)
X_nmf = nmf.fit_transform(X_mm)
print(f"  NMF con {N_COMPONENTS_PCA} componentes")
print(f"  Error de reconstrucción: {nmf.reconstruction_err_:.2f}")

# Top features por tema NMF
print(f"\n  Temas NMF (top features por componente):")
for i in range(min(5, N_COMPONENTS_PCA)):
    loadings = pd.Series(nmf.components_[i], index=feature_names)
    top = loadings.nlargest(5)
    print(f"\n  Tema {i+1}:")
    for n, v in top.items():
        print(f"    {n:45s} {v:.3f}")


Método A2: Non-Negative Matrix Factorization (NMF)
  NMF con 29 componentes
  Error de reconstrucción: 7.34

  Temas NMF (top features por componente):

  Tema 1:
    hora_tarde                                    8.095
    trim_Q4                                       7.360
    dia_saturday                                  3.824
    dia_wednesday                                 2.654
    delito_fraude_y_delitos_patrimoniales         1.866

  Tema 2:
    delito_falsificacion_y_documentos             6.217
    trim_Q4                                       0.903
    dia_friday                                    0.875
    intensidad_log                                0.796
    dia_wednesday                                 0.631

  Tema 3:
    delito_robo_con_violencia                     7.226
    ratio_violencia                               5.879
    dia_tuesday                                   1.787
    dia_friday                                    1.649
    hora_noche                

In [7]:
# ============================================================================
# PASO 2C: Clustering — K-Means
# ============================================================================
print(f"\n{'='*80}")
print(f"Método A3: K-Means (sobre embeddings PCA)")
print(f"{'='*80}")

# Buscar K óptimo
resultados_k = []
for k in range(3, 16):
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_pca)
    sil = silhouette_score(X_pca, labels)
    ch = calinski_harabasz_score(X_pca, labels)
    inertia = km.inertia_
    resultados_k.append({'k': k, 'silhouette': sil, 'calinski_harabasz': ch, 'inertia': inertia})
    print(f"  K={k:2d}  Silhouette={sil:.3f}  Calinski-Harabasz={ch:.1f}")

# Elegir mejor K por silhouette
df_k = pd.DataFrame(resultados_k)
mejor_k = df_k.loc[df_k['silhouette'].idxmax(), 'k']
print(f"\n  Mejor K por Silhouette: {int(mejor_k)}\n")

# Entrenar K-Means final
km_final = KMeans(n_clusters=int(mejor_k), n_init=20, random_state=42)
clusters_km = km_final.fit_predict(X_pca)
print(f"  Distribución de clusters K-Means:")
for c, count in pd.Series(clusters_km).value_counts().sort_index().items():
    print(f"    Cluster {c}: {count} hexágonos ({count/len(clusters_km)*100:.1f}%)")


Método A3: K-Means (sobre embeddings PCA)
  K= 3  Silhouette=0.107  Calinski-Harabasz=120.2
  K= 4  Silhouette=0.112  Calinski-Harabasz=96.4
  K= 5  Silhouette=0.060  Calinski-Harabasz=84.3
  K= 6  Silhouette=0.072  Calinski-Harabasz=76.8
  K= 7  Silhouette=0.072  Calinski-Harabasz=68.5
  K= 8  Silhouette=0.049  Calinski-Harabasz=63.4
  K= 9  Silhouette=0.061  Calinski-Harabasz=59.8
  K=10  Silhouette=0.040  Calinski-Harabasz=54.8
  K=11  Silhouette=0.066  Calinski-Harabasz=52.3
  K=12  Silhouette=0.061  Calinski-Harabasz=50.0
  K=13  Silhouette=0.038  Calinski-Harabasz=47.1
  K=14  Silhouette=0.026  Calinski-Harabasz=45.9
  K=15  Silhouette=0.035  Calinski-Harabasz=43.8

  Mejor K por Silhouette: 4

  Distribución de clusters K-Means:
    Cluster 0: 302 hexágonos (28.5%)
    Cluster 1: 270 hexágonos (25.4%)
    Cluster 2: 41 hexágonos (3.9%)
    Cluster 3: 448 hexágonos (42.2%)


In [8]:
# ============================================================================
# PASO 2D: GMM
# ============================================================================
print(f"\n{'='*80}")
print(f"Método A4: Gaussian Mixture Model (GMM)")
print(f"{'='*80}")

resultados_gmm = []
for k in range(3, 16):
    gmm = GaussianMixture(n_components=k, covariance_type='full', random_state=42, n_init=3)
    labels = gmm.fit_predict(X_pca)
    sil = silhouette_score(X_pca, labels)
    bic = gmm.bic(X_pca)
    aic = gmm.aic(X_pca)
    resultados_gmm.append({'k': k, 'silhouette': sil, 'bic': bic, 'aic': aic})
    print(f"  K={k:2d}  Silhouette={sil:.3f}  BIC={bic:.0f}  AIC={aic:.0f}")

df_gmm = pd.DataFrame(resultados_gmm)
mejor_k_gmm = df_gmm.loc[df_gmm['bic'].idxmin(), 'k']
print(f"\n  Mejor K por BIC: {int(mejor_k_gmm)}")

gmm_final = GaussianMixture(n_components=int(mejor_k_gmm), covariance_type='full',
                             random_state=42, n_init=5)
clusters_gmm = gmm_final.fit_predict(X_pca)
#probs_gmm = gmm_final.predict_proba(X_pca)
# TODO: podemos usar estas probabilidades también para un analísis más fino?


Método A4: Gaussian Mixture Model (GMM)
  K= 3  Silhouette=0.145  BIC=80405  AIC=73481
  K= 4  Silhouette=0.110  BIC=79404  AIC=70171
  K= 5  Silhouette=0.086  BIC=82901  AIC=71358
  K= 6  Silhouette=0.060  BIC=83067  AIC=69214
  K= 7  Silhouette=0.024  BIC=84606  AIC=68443
  K= 8  Silhouette=0.064  BIC=81637  AIC=63165
  K= 9  Silhouette=0.078  BIC=87492  AIC=66710
  K=10  Silhouette=0.033  BIC=88628  AIC=65537
  K=11  Silhouette=0.069  BIC=90246  AIC=64845
  K=12  Silhouette=0.022  BIC=90634  AIC=62923
  K=13  Silhouette=0.055  BIC=94333  AIC=64313
  K=14  Silhouette=0.051  BIC=94064  AIC=61734
  K=15  Silhouette=0.017  BIC=95156  AIC=60516

  Mejor K por BIC: 4


In [9]:
# ============================================================================
# PASO 2E: HDBSCAN
# ============================================================================
print(f"\n{'='*80}")
print(f"Método A5: HDBSCAN")
print(f"{'='*80}")

# Análisis de sensibilidad: encontrar parámetros óptimos
best_result = None
best_n_clusters = 0

for min_size in [5, 10, 15, 20]:
    for min_samples in [2, 3, 5]:
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=min_size,
            min_samples=min_samples,
            metric='euclidean',
            cluster_selection_method='leaf'  # 'leaf' es más permisivo que 'eom'
        )
        labels = clusterer.fit_predict(X_pca)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = (labels == -1).sum()
        
        # Preferir configuraciones que encuentren clusters (no ruido puro)
        if n_clusters > best_n_clusters:
            best_n_clusters = n_clusters
            best_result = {
                'min_size': min_size,
                'min_samples': min_samples,
                'labels': labels,
                'n_clusters': n_clusters,
                'n_noise': n_noise
            }
        
        if n_clusters > 0:
            print(f"    min_size={min_size}, min_samples={min_samples}: "
                    f"{n_clusters} clusters, {n_noise} ruido ({n_noise/len(labels)*100:.1f}%)")

if best_result is None:
    print(f"No se encontraron clusters. Reducción de cluster_size y min_samples.")
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=5,
        min_samples=1,
        metric='euclidean',
        cluster_selection_method='leaf'
    )
    clusters_hdb = clusterer.fit_predict(X_pca)
else:
    print(f"\n  Mejor configuración: min_size={best_result['min_size']}, "
            f"min_samples={best_result['min_samples']}")
    clusters_hdb = best_result['labels']

n_clusters_hdb = len(set(clusters_hdb)) - (1 if -1 in clusters_hdb else 0)
n_noise = (clusters_hdb == -1).sum()

print(f"\n  Clusters encontrados: {n_clusters_hdb}")
print(f"  Puntos de ruido: {n_noise} ({n_noise/len(clusters_hdb)*100:.1f}%)")

if n_clusters_hdb > 1:
    mask_valid = clusters_hdb != -1
    if mask_valid.sum() > 0:
        sil_hdb = silhouette_score(X_pca[mask_valid], clusters_hdb[mask_valid])
        print(f"  Silhouette (sin ruido): {sil_hdb:.3f}")

print(f"  Distribución:")
for c, count in pd.Series(clusters_hdb).value_counts().sort_index().items():
    label = f"Cluster {c}" if c >= 0 else "Ruido"
    print(f"    {label}: {count} hexágonos ({count/len(clusters_hdb)*100:.1f}%)")



Método A5: HDBSCAN
    min_size=5, min_samples=2: 4 clusters, 948 ruido (89.3%)
    min_size=5, min_samples=3: 5 clusters, 1025 ruido (96.6%)
    min_size=10, min_samples=2: 3 clusters, 955 ruido (90.0%)

  Mejor configuración: min_size=5, min_samples=3

  Clusters encontrados: 5
  Puntos de ruido: 1025 (96.6%)
  Silhouette (sin ruido): 0.336
  Distribución:
    Ruido: 1025 hexágonos (96.6%)
    Cluster 0: 7 hexágonos (0.7%)
    Cluster 1: 7 hexágonos (0.7%)
    Cluster 2: 9 hexágonos (0.8%)
    Cluster 3: 7 hexágonos (0.7%)
    Cluster 4: 6 hexágonos (0.6%)


# TERMINA ADOLFO

# EMPIEZA ARA

In [25]:

# ============================================================================
# PASO 3: AUTOENCODER DENSO
# ============================================================================
print(f"\n{'='*80}")
print(f"Método B: Autoencoder denso")
print(f"{'='*80}")

# Arquitectura
INPUT_DIM = X_std.shape[1]  # 45
ENCODING_DIM = 12  # embedding de salida
HIDDEN_DIMS = [32, 24]  # capas intermedias

class CrimeAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dims, encoding_dim):
        super().__init__()
        
        # Encoder
        encoder_layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            encoder_layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.BatchNorm1d(h_dim),
                nn.ReLU(),
                nn.Dropout(0.1)
            ])
            prev_dim = h_dim
        encoder_layers.append(nn.Linear(prev_dim, encoding_dim))
        self.encoder = nn.Sequential(*encoder_layers)
        
        # Decoder (simétrico)
        decoder_layers = []
        prev_dim = encoding_dim
        for h_dim in reversed(hidden_dims):
            decoder_layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.BatchNorm1d(h_dim),
                nn.ReLU(),
                nn.Dropout(0.1)
            ])
            prev_dim = h_dim
        decoder_layers.append(nn.Linear(prev_dim, input_dim))
        self.decoder = nn.Sequential(*decoder_layers)
    
    def encode(self, x):
        return self.encoder(x)
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        z = self.encode(x)
        x_recon = self.decode(z)
        return x_recon, z

# Preparar datos
X_tensor = torch.tensor(X_std, dtype=torch.float32, device=device).to(device) # NOTE: ,  dtype y device son configuraciones para que use el gpu si está disponible
dataset = TensorDataset(X_tensor, X_tensor)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# Entrenar
model = CrimeAutoencoder(INPUT_DIM, HIDDEN_DIMS, ENCODING_DIM).to(device) # NOTE configuración para usar gpu
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=20, factor=0.5)
criterion = nn.MSELoss()

EPOCHS = 300
best_loss = float('inf')
patience_counter = 0
PATIENCE = 50

print(f"  Arquitectura: {INPUT_DIM} → {HIDDEN_DIMS} → {ENCODING_DIM} → {list(reversed(HIDDEN_DIMS))} → {INPUT_DIM}")
print(f"  Entrenando ({EPOCHS} epochs max, early stopping patience={PATIENCE})...")

# guardar loss autoencoder para análisis posterior
losses_ae = []
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    for batch_x, _ in dataloader:
        optimizer.zero_grad()
        x_recon, z = model(batch_x)
        loss = criterion(x_recon, batch_x)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(dataloader)
    losses_ae.append(avg_loss)
    scheduler.step(avg_loss)
    
    if avg_loss < best_loss:
        best_loss = avg_loss
        patience_counter = 0
        best_state = model.state_dict().copy()
    else:
        patience_counter += 1
    
    if patience_counter >= PATIENCE:
        print(f"  Early stopping en epoch {epoch+1}")
        break
    
    if (epoch + 1) % 50 == 0:
        print(f"    Epoch {epoch+1:3d}: loss = {avg_loss:.6f}")

# Cargar mejor modelo
model.load_state_dict(best_state)
print(f"  Mejor loss: {best_loss:.6f}")

# Extraer embeddings
model.eval()
with torch.no_grad():
    x_recon, embeddings_ae = model(X_tensor)
    recon_error = torch.mean((X_tensor - x_recon) ** 2, dim=1).cpu().numpy()

X_ae = embeddings_ae.cpu().numpy()
print(f"  Embeddings shape: {X_ae.shape}")
print(f"  Error de reconstrucción medio: {recon_error.mean():.4f}")
print(f"  Error de reconstrucción máximo: {recon_error.max():.4f}")

# Guardar modelo
torch.save(best_state, '../models/modelo_autoencoder.pth')
print(f"  Modelo guardado: ../models/modelo_autoencoder.pth")



Método B: Autoencoder denso
  Arquitectura: 45 → [32, 24] → 12 → [24, 32] → 45
  Entrenando (300 epochs max, early stopping patience=50)...
    Epoch  50: loss = 0.635090
    Epoch 100: loss = 0.601745
    Epoch 150: loss = 0.578702
    Epoch 200: loss = 0.571792
    Epoch 250: loss = 0.560128
  Early stopping en epoch 291
  Mejor loss: 0.557519
  Embeddings shape: (1061, 12)
  Error de reconstrucción medio: 0.4583
  Error de reconstrucción máximo: 4.8069
  Modelo guardado: ../models/modelo_autoencoder.pth


# TERMINA ARA

# EMPIEZA NICOLE

In [26]:
# ============================================================================
# PASO 4: CONSTRUIR GRAFO DE VECINDAD H3
# ============================================================================
print(f"\n{'='*80}")
print(f"Grafo de vecindad H3")
print(f"{'='*80}")

# Mapear h3_id a índice numérico
h3_to_idx = {h: i for i, h in enumerate(h3_ids)}

# Construir aristas: cada hexágono conectado a sus vecinos (k-ring=1)
edges_src, edges_dst = [], []
for h3_id in h3_ids:
    vecinos = h3.grid_disk(h3_id, 1)  # incluye el propio hexágono
    for vecino in vecinos:
        if vecino != h3_id and vecino in h3_to_idx:
            edges_src.append(h3_to_idx[h3_id])
            edges_dst.append(h3_to_idx[vecino])

edge_index = torch.tensor([edges_src, edges_dst], dtype=torch.long).to(device)
n_edges = edge_index.shape[1]
avg_degree = n_edges / len(h3_ids)

print(f"  Nodos: {len(h3_ids)}")
print(f"  Aristas: {n_edges}")
print(f"  Grado promedio: {avg_degree:.1f}")
print(f"  Componentes conexas: verificando...")

# Verificar conectividad con BFS simple
from collections import deque
def count_components(n_nodes, src, dst):
    adj = {i: [] for i in range(n_nodes)}
    for s, d in zip(src, dst):
        adj[s].append(d)
    visited = set()
    components = 0
    for start in range(n_nodes):
        if start in visited:
            continue
        components += 1
        queue = deque([start])
        while queue:
            node = queue.popleft()
            if node in visited:
                continue
            visited.add(node)
            for neighbor in adj[node]:
                if neighbor not in visited:
                    queue.append(neighbor)
    return components

n_comp = count_components(len(h3_ids), edges_src, edges_dst)
print(f"  Componentes conexas: {n_comp}")

# También añadir conexiones k-ring=2 para hexágonos aislados
if n_comp > 1:
    print(f"  Añadiendo vecinos k-ring=2 para conectar componentes...")
    edges_src2, edges_dst2 = edges_src.copy(), edges_dst.copy()
    for h3_id in h3_ids:
        vecinos_k2 = h3.grid_disk(h3_id, 2)
        for vecino in vecinos_k2:
            if vecino != h3_id and vecino in h3_to_idx:
                edges_src2.append(h3_to_idx[h3_id])
                edges_dst2.append(h3_to_idx[vecino])
    # Deduplicar
    edge_set = set(zip(edges_src2, edges_dst2))
    edges_src2, edges_dst2 = zip(*edge_set) if edge_set else ([], [])
    edges_src2, edges_dst2 = list(edges_src2), list(edges_dst2)

    n_comp2 = count_components(len(h3_ids), edges_src2, edges_dst2)
    print(f"  Componentes con k-ring=2: {n_comp2}")
    print(f"  Aristas con k-ring=2: {len(edges_src2)}")

    edge_index = torch.tensor([edges_src2, edges_dst2], dtype=torch.long).to(device)


Grafo de vecindad H3
  Nodos: 1061
  Aristas: 5722
  Grado promedio: 5.4
  Componentes conexas: verificando...
  Componentes conexas: 2
  Añadiendo vecinos k-ring=2 para conectar componentes...
  Componentes con k-ring=2: 2
  Aristas con k-ring=2: 16390


In [27]:
# ============================================================================
# PASO 5: GRAPH AUTOENCODER (GAE)
# ============================================================================
print(f"\n{'='*80}")
print(f"Método C: Graph Autoencoder (GAE)")
print(f"{'='*80}")

X_tensor_graph = torch.FloatTensor(X_std).to(device)

class GraphEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.conv3 = GCNConv(hidden_dim, out_dim)

    def forward(self, x, edge_index):
        h = self.conv1(x, edge_index)
        h = self.bn1(h)
        h = torch.relu(h)
        h = self.conv2(h, edge_index)
        h = self.bn2(h)
        h = torch.relu(h)
        z = self.conv3(h, edge_index)
        return z

class GAE(nn.Module):
    """Graph Autoencoder: encoder GCN + decoder por producto interno."""
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder

    def encode(self, x, edge_index):
        return self.encoder(x, edge_index)

    def decode(self, z, edge_index_decode):
        """Reconstruye aristas como producto interno entre embeddings."""
        src, dst = edge_index_decode
        return (z[src] * z[dst]).sum(dim=1)

    def recon_loss(self, z, pos_edge_index, neg_edge_index):
        """Binary cross-entropy entre aristas positivas y negativas."""
        pos_score = self.decode(z, pos_edge_index)
        neg_score = self.decode(z, neg_edge_index)
        pos_loss = -torch.log(torch.sigmoid(pos_score) + 1e-8).mean()
        neg_loss = -torch.log(1 - torch.sigmoid(neg_score) + 1e-8).mean()
        return pos_loss + neg_loss

GAE_HIDDEN = 32
GAE_OUT = 12

encoder_gae = GraphEncoder(INPUT_DIM, GAE_HIDDEN, GAE_OUT).to(device)
model_gae = GAE(encoder_gae).to(device)
optimizer_gae = torch.optim.Adam(model_gae.parameters(), lr=0.005, weight_decay=1e-5)

EPOCHS_GAE = 300
PATIENCE_GAE = 50
print(f"  Arquitectura: {INPUT_DIM} → GCN({GAE_HIDDEN}) → GCN({GAE_HIDDEN}) → GCN({GAE_OUT})")
print(f"  Entrenando GAE...")

best_loss_gae = float('inf')
patience_gae = 0
losses_gae = []

for epoch in range(EPOCHS_GAE):
    model_gae.train()
    optimizer_gae.zero_grad()

    z = model_gae.encode(X_tensor_graph, edge_index)

    # Negative sampling
    neg_edge = negative_sampling(
        edge_index=edge_index,
        num_nodes=len(h3_ids),
        num_neg_samples=edge_index.size(1)
    )

    loss = model_gae.recon_loss(z, edge_index, neg_edge)
    loss.backward()
    optimizer_gae.step()

    l = loss.item()
    losses_gae.append(l)

    if l < best_loss_gae:
        best_loss_gae = l
        patience_gae = 0
        best_state_gae = model_gae.state_dict().copy()
    else:
        patience_gae += 1

    if patience_gae >= PATIENCE_GAE:
        print(f"  Early stopping epoch {epoch+1}")
        break

    if (epoch+1) % 50 == 0:
        print(f"    Epoch {epoch+1}: loss={l:.4f}")

model_gae.load_state_dict(best_state_gae)
model_gae.eval()
with torch.no_grad():
    X_gae = model_gae.encode(X_tensor_graph, edge_index).cpu().numpy()

print(f"  Mejor loss: {best_loss_gae:.4f}")
print(f"  Embeddings GAE: {X_gae.shape}")
torch.save(best_state_gae, '../models/modelo_gae.pth')
print(f"  /models/modelo_gae.pth")


Método C: Graph Autoencoder (GAE)
  Arquitectura: 45 → GCN(32) → GCN(32) → GCN(12)
  Entrenando GAE...
    Epoch 50: loss=0.8297
    Epoch 100: loss=0.8052
    Epoch 150: loss=0.8102
    Epoch 200: loss=0.7973
    Epoch 250: loss=0.8096
    Epoch 300: loss=0.8033
  Mejor loss: 0.7866
  Embeddings GAE: (1061, 12)
  /models/modelo_gae.pth


In [28]:
# ============================================================================
# PASO 6: VARIATIONAL GRAPH AUTOENCODER (VGAE)
# ============================================================================
print(f"\n{'='*80}")
print(f"Método D: Variational Graph Autoencoder (VGAE)")
print(f"{'='*80}")

class VariationalGraphEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        # Dos cabezas: mu y logvar
        self.conv_mu = GCNConv(hidden_dim, out_dim)
        self.conv_logvar = GCNConv(hidden_dim, out_dim)

    def forward(self, x, edge_index):
        h = self.conv1(x, edge_index)
        h = self.bn1(h)
        h = torch.relu(h)
        h = self.conv2(h, edge_index)
        h = self.bn2(h)
        h = torch.relu(h)
        mu = self.conv_mu(h, edge_index)
        logvar = self.conv_logvar(h, edge_index)
        return mu, logvar

class VGAE(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder

    def encode(self, x, edge_index):
        mu, logvar = self.encoder(x, edge_index)
        return mu, logvar

    def reparametrize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu

    def decode(self, z, edge_index_decode):
        src, dst = edge_index_decode
        return (z[src] * z[dst]).sum(dim=1)

    def recon_loss(self, z, pos_edge_index, neg_edge_index):
        pos_score = self.decode(z, pos_edge_index)
        neg_score = self.decode(z, neg_edge_index)
        pos_loss = -torch.log(torch.sigmoid(pos_score) + 1e-8).mean()
        neg_loss = -torch.log(1 - torch.sigmoid(neg_score) + 1e-8).mean()
        return pos_loss + neg_loss

    def kl_loss(self, mu, logvar):
        return -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

VGAE_HIDDEN = 32
VGAE_OUT = 12

encoder_vgae = VariationalGraphEncoder(INPUT_DIM, VGAE_HIDDEN, VGAE_OUT).to(device)
model_vgae = VGAE(encoder_vgae).to(device)
optimizer_vgae = torch.optim.Adam(model_vgae.parameters(), lr=0.005, weight_decay=1e-5)

print(f"  Arquitectura: {INPUT_DIM} → GCN({VGAE_HIDDEN}) → GCN({VGAE_HIDDEN}) → [μ({VGAE_OUT}), σ({VGAE_OUT})]")
print(f"  Entrenando VGAE...")

KL_WEIGHT = 0.001  # Peso bajo para KL — evita colapso posterior
best_loss_vgae = float('inf')
patience_vgae = 0
losses_vgae = []

for epoch in range(EPOCHS_GAE):
    model_vgae.train()
    optimizer_vgae.zero_grad()

    mu, logvar = model_vgae.encode(X_tensor_graph, edge_index)
    z = model_vgae.reparametrize(mu, logvar)

    neg_edge = negative_sampling(
        edge_index=edge_index,
        num_nodes=len(h3_ids),
        num_neg_samples=edge_index.size(1)
    )

    loss_recon = model_vgae.recon_loss(z, edge_index, neg_edge)
    loss_kl = model_vgae.kl_loss(mu, logvar)
    loss = loss_recon + KL_WEIGHT * loss_kl
    loss.backward()
    optimizer_vgae.step()

    l = loss.item()
    losses_vgae.append(l)

    if l < best_loss_vgae:
        best_loss_vgae = l
        patience_vgae = 0
        best_state_vgae = model_vgae.state_dict().copy()
    else:
        patience_vgae += 1

    if patience_vgae >= PATIENCE_GAE:
        print(f"  Early stopping epoch {epoch+1}")
        break

    if (epoch+1) % 50 == 0:
        print(f"    Epoch {epoch+1}: loss={l:.4f} (recon={loss_recon.item():.4f}, kl={loss_kl.item():.4f})")

model_vgae.load_state_dict(best_state_vgae)
model_vgae.eval()
with torch.no_grad():
    mu_final, logvar_final = model_vgae.encode(X_tensor_graph, edge_index)
    X_vgae = mu_final.cpu().numpy()  # Usar mu como embedding (más estable que sampling)

print(f"  Mejor loss: {best_loss_vgae:.4f}")
print(f"  Embeddings VGAE: {X_vgae.shape}")
torch.save(best_state_vgae, '../models/modelo_vgae.pth')
print(f"  /models/modelo_vgae.pth")


Método D: Variational Graph Autoencoder (VGAE)
  Arquitectura: 45 → GCN(32) → GCN(32) → [μ(12), σ(12)]
  Entrenando VGAE...
    Epoch 50: loss=0.9822 (recon=0.9813, kl=0.8432)
    Epoch 100: loss=0.8732 (recon=0.8719, kl=1.2870)
    Epoch 150: loss=0.8353 (recon=0.8337, kl=1.5735)
    Epoch 200: loss=0.8297 (recon=0.8279, kl=1.8019)
    Epoch 250: loss=0.8144 (recon=0.8124, kl=1.9835)
  Early stopping epoch 269
  Mejor loss: 0.8075
  Embeddings VGAE: (1061, 12)
  /models/modelo_vgae.pth


# TERMINA NICOLE

# EMPIEZA LUISMI

In [29]:
# ============================================================================
# PASO 7: CLUSTERING SOBRE TODOS LOS EMBEDDINGS
# ============================================================================
print(f"\n{'='*80}")
print(f"Clustering sobre todos los embeddings")
print(f"{'='*80}")

all_embeddings = {
    'PCA': X_pca,
    'Autoencoder': X_ae,
    'GAE': X_gae,
    'VGAE': X_vgae,
}

best_clusters = {}
for nombre, X_emb in all_embeddings.items():
    best_sil = -1
    best_k_local = 3
    for k in range(3, 16):
        km = KMeans(n_clusters=k, n_init=10, random_state=42)
        labels = km.fit_predict(X_emb)
        sil = silhouette_score(X_emb, labels)
        if sil > best_sil:
            best_sil = sil
            best_k_local = k
    km = KMeans(n_clusters=best_k_local, n_init=20, random_state=42)
    best_clusters[nombre] = km.fit_predict(X_emb)
    print(f"  {nombre:15s}  Mejor K={best_k_local:2d}  Silhouette={best_sil:.3f}")

clusters_ae = best_clusters['Autoencoder']


Clustering sobre todos los embeddings
  PCA              Mejor K= 4  Silhouette=0.112
  Autoencoder      Mejor K= 3  Silhouette=0.206
  GAE              Mejor K=15  Silhouette=0.357
  VGAE             Mejor K=15  Silhouette=0.325


In [34]:
# ============================================================================
# PASO 8: COMPARACIÓN DE REPRESENTACIONES
# ============================================================================
print(f"\n{'='*80}")
print(f"Comparación de representaciones")
print(f"{'='*80}")

print(f"\n  {'Método':<15s} {'Dims':>5s} {'Best K':>7s} {'Silhouette':>11s}")
print(f"  {'-'*42}")
for nombre, X_emb in all_embeddings.items():
    labels = best_clusters[nombre]
    n_cl = len(set(labels))
    sil = silhouette_score(X_emb, labels)
    print(f"  {nombre:<15s} {X_emb.shape[1]:>5d} {n_cl:>7d} {sil:>11.3f}")

print(f"\n  Concordancia entre métodos (ARI / NMI):")
nombres_metodos = list(best_clusters.keys())
# Incluir GMM y HDBSCAN
all_cluster_labels = dict(best_clusters)
all_cluster_labels['GMM'] = clusters_gmm
all_cluster_labels['HDBSCAN'] = clusters_hdb
nombres_metodos = list(all_cluster_labels.keys())

for i in range(len(nombres_metodos)):
    for j in range(i+1, len(nombres_metodos)):
        n1, n2 = nombres_metodos[i], nombres_metodos[j]
        ari = adjusted_rand_score(all_cluster_labels[n1], all_cluster_labels[n2])
        nmi = normalized_mutual_info_score(all_cluster_labels[n1], all_cluster_labels[n2])
        if ari > 0.3:  # Solo mostrar pares con concordancia notable
            print(f"    {n1:15s} vs {n2:15s}  ARI={ari:.3f}  NMI={nmi:.3f}")


Comparación de representaciones

  Método           Dims  Best K  Silhouette
  ------------------------------------------
  PCA                29       4       0.112
  Autoencoder        12       3       0.207
  GAE                12      15       0.357
  VGAE               12      15       0.319

  Concordancia entre métodos (ARI / NMI):
    PCA             vs Autoencoder      ARI=0.563  NMI=0.572
    PCA             vs GMM              ARI=0.380  NMI=0.367
    Autoencoder     vs GMM              ARI=0.544  NMI=0.458
    GAE             vs VGAE             ARI=0.529  NMI=0.720


In [31]:
# ============================================================================
# PASO 9: PERFILES DE CLUSTERS (VGAE como representación principal)
# ============================================================================
# TODO justificar por qué VGAE es la representación principal (mejor silhouette, más interpretabilidad, etc.)
print(f"\n{'='*80}")
print(f"Perfiles de clusters (VGAE + K-Means)")
print(f"{'='*80}")

clusters_vgae = best_clusters['VGAE']
firmas_temp = firmas.copy()
firmas_temp['cluster_vgae'] = clusters_vgae
firmas_con_meta = firmas_temp.join(metadata)

for c in sorted(firmas_temp['cluster_vgae'].unique()):
    grupo = firmas_con_meta[firmas_con_meta['cluster_vgae'] == c]
    n = len(grupo)
    print(f"\n▸ Cluster {c} — {n} hexágonos ({n/len(firmas)*100:.1f}%)")

    alcs = grupo['alcaldia_dominante'].value_counts().head(3)
    print(f"  Alcaldías: {', '.join([f'{a} ({ct})' for a, ct in alcs.items()])}")

    delito_cols = [col for col in firmas.columns if col.startswith('delito_')]
    top_delitos = grupo[delito_cols].mean().sort_values(ascending=False).head(5)
    print(f"  Perfil delictivo:")
    for col, val in top_delitos.items():
        nombre = col.replace('delito_', '').replace('_', ' ').upper()
        print(f"    {nombre:40s} {val*100:5.1f}%")

    print(f"  Intensidad: {grupo['intensidad_log'].mean():.2f} (~{np.expm1(grupo['intensidad_log'].mean()):.0f} registros)")
    print(f"  Ratio violencia: {grupo['ratio_violencia'].mean():.3f}")


Perfiles de clusters (VGAE + K-Means)

▸ Cluster 0 — 90 hexágonos (8.5%)
  Alcaldías: AZCAPOTZALCO (46), MIGUEL HIDALGO (36), ALVARO OBREGON (4)
  Perfil delictivo:
    ROBO SIN VIOLENCIA                        30.1%
    ROBO CON VIOLENCIA                        14.9%
    FRAUDE Y DELITOS PATRIMONIALES            14.5%
    VIOLENCIA FAMILIAR                        10.5%
    AMENAZAS                                   6.5%
  Intensidad: 7.32 (~1514 registros)
  Ratio violencia: 0.178

▸ Cluster 1 — 75 hexágonos (7.1%)
  Alcaldías: IZTAPALAPA (40), IZTACALCO (23), VENUSTIANO CARRANZA (12)
  Perfil delictivo:
    ROBO SIN VIOLENCIA                        26.6%
    VIOLENCIA FAMILIAR                        16.1%
    ROBO CON VIOLENCIA                        14.1%
    FRAUDE Y DELITOS PATRIMONIALES            11.4%
    AMENAZAS                                   7.3%
  Intensidad: 7.64 (~2082 registros)
  Ratio violencia: 0.171

▸ Cluster 2 — 59 hexágonos (5.6%)
  Alcaldías: TLALPAN (51), CU

# TERMINA LUISMI

# EMPIEZA ROCHA

In [32]:
# ============================================================================
# PASO 9.1: Perfiles de clusters (usando Autoencoder como representación principal)
# ============================================================================
# TODO comparar perfiles entre VGAE y Autoencoder — ¿son similares? ¿qué diferencias hay?
print(f"\n{'='*80}")
print(f"Perfiles de clusters (Autoencoder + K-Means)")
print(f"{'='*80}")

firmas['cluster_ae'] = clusters_ae
firmas_con_meta = firmas.join(metadata)

for c in sorted(firmas['cluster_ae'].unique()):
    grupo = firmas_con_meta[firmas_con_meta['cluster_ae'] == c]
    n = len(grupo)
    print(f"\n- Cluster {c} — {n} hexágonos ({n/len(firmas)*100:.1f}%)")
    
    # Alcaldías dominantes
    alcs = grupo['alcaldia_dominante'].value_counts().head(3)
    print(f"  Alcaldías: {', '.join([f'{a} ({c_})' for a, c_ in alcs.items()])}")
    
    # Top delitos
    delito_cols = [col for col in firmas.columns if col.startswith('delito_')]
    top_delitos = grupo[delito_cols].mean().sort_values(ascending=False).head(5)
    print(f"  Perfil delictivo:")
    for col, val in top_delitos.items():
        nombre = col.replace('delito_', '').replace('_', ' ').upper()
        print(f"    {nombre:40s} {val*100:5.1f}%")
    
    # Intensidad y violencia
    print(f"  Intensidad media: {grupo['intensidad_log'].mean():.2f} (~{np.expm1(grupo['intensidad_log'].mean()):.0f} registros)")
    print(f"  Ratio violencia: {grupo['ratio_violencia'].mean():.3f}")

# Limpiar columna temporal
firmas.drop('cluster_ae', axis=1, inplace=True)


Perfiles de clusters (Autoencoder + K-Means)

- Cluster 0 — 583 hexágonos (54.9%)
  Alcaldías: IZTAPALAPA (131), GUSTAVO A. MADERO (85), TLALPAN (51)
  Perfil delictivo:
    ROBO SIN VIOLENCIA                        24.3%
    VIOLENCIA FAMILIAR                        17.2%
    ROBO CON VIOLENCIA                        13.2%
    FRAUDE Y DELITOS PATRIMONIALES            11.9%
    AMENAZAS                                   7.9%
  Intensidad media: 7.41 (~1651 registros)
  Ratio violencia: 0.167

- Cluster 1 — 249 hexágonos (23.5%)
  Alcaldías: TLALPAN (57), MILPA ALTA (52), XOCHIMILCO (42)
  Perfil delictivo:
    VIOLENCIA FAMILIAR                        25.4%
    ROBO SIN VIOLENCIA                        14.7%
    FRAUDE Y DELITOS PATRIMONIALES            11.7%
    AMENAZAS                                   9.1%
    ROBO CON VIOLENCIA                         8.0%
  Intensidad media: 5.40 (~221 registros)
  Ratio violencia: 0.138

- Cluster 2 — 229 hexágonos (21.6%)
  Alcaldías: MIGUEL 

In [33]:
# ============================================================================
# PASO 10: Exportar embeddings y clusters
# ============================================================================

# Embeddings (autoencoder como principal, PCA como línea base)
embeddings_df = pd.DataFrame(
    X_ae,
    index=firmas.index,
    columns=[f'emb_ae_{i}' for i in range(X_ae.shape[1])]
)
# Añadir PCA también
for i in range(X_pca.shape[1]):
    embeddings_df[f'emb_pca_{i}'] = X_pca[:, i]

# Añadir GAE también
for i in range(X_gae.shape[1]):
    embeddings_df[f'emb_gae_{i}'] = X_gae[:, i]

# Añadir VGAE también
for i in range(X_vgae.shape[1]):
    embeddings_df[f'emb_vgae_{i}'] = X_vgae[:, i]

# Añadir error de reconstrucción (útil para la detección de disparidad final)
embeddings_df['recon_error'] = recon_error

embeddings_df.index.name = 'h3_id'
embeddings_df.to_csv('../data/results/embeddings_h3.csv', encoding='utf-8-sig')
print(f"../data/results/embeddings_h3.csv ({len(embeddings_df)} hexágonos × {embeddings_df.shape[1]} columnas)")

# Clusters
clusters_df = pd.DataFrame({
    'cluster_kmeans_pca': clusters_km,
    'cluster_gmm_pca': clusters_gmm,
    'cluster_hdbscan': clusters_hdb,
    'cluster_kmeans_ae': best_clusters['Autoencoder'],
    'cluster_kmeans_gae': best_clusters['GAE'],
    'cluster_kmeans_vgae': best_clusters['VGAE'],
}, index=firmas.index)
clusters_df.index.name = 'h3_id'
clusters_df.to_csv('../data/auxiliar/clusters_h3.csv', encoding='utf-8-sig')
print(f"../data/auxiliar/clusters_h3.csv ({len(clusters_df)} hexágonos × {clusters_df.shape[1]} métodos)")

# Métricas de entrenamiento
# pd.DataFrame({'epoch': range(1, len(losses)+1), 'loss': losses}).to_csv(
#     '../data/results/autoencoder_training_loss.csv', index=False
# )
# print(f"../data/results/autoencoder_training_loss.csv ({len(losses)} epochs)")

# Losses de entrenamiento
losses_df = pd.DataFrame({
    'epoch': range(1, max(len(losses_ae), len(losses_gae), len(losses_vgae)) + 1),
    'loss_ae': losses_ae + [None]*(max(len(losses_ae), len(losses_gae), len(losses_vgae)) - len(losses_ae)),
    'loss_gae': losses_gae + [None]*(max(len(losses_ae), len(losses_gae), len(losses_vgae)) - len(losses_gae)),
    'loss_vgae': losses_vgae + [None]*(max(len(losses_ae), len(losses_gae), len(losses_vgae)) - len(losses_vgae)),
})
losses_df.to_csv('../data/results/training_losses.csv', index=False)
print(f"../data/results/training_losses.csv")


../data/results/embeddings_h3.csv (1061 hexágonos × 66 columnas)
../data/auxiliar/clusters_h3.csv (1061 hexágonos × 6 métodos)
../data/results/training_losses.csv


# TERMINA ROCHA

# EMPIEZA ARA PT2

### Ablation study
Objetivo: Selección rigurosa de hiperparámetros para GAE/VGAE
          con validación robusta, reproducible y publicable.

Ablation sobre:
  - Dimensión del embedding: [8, 12, 16]
  - Profundidad GCN: [2, 3]
  - Hidden dimension: [32, 48]
  - Modelo: [GAE, VGAE]

Validación:
  - 5 seeds por configuración (estabilidad)
  - Métricas: Silhouette, Calinski-Harabasz, link prediction AUC/AP
  - HDBSCAN sobre embeddings óptimos
  - Reporte completo para paper

Input:  firmas_h3.csv, h3_metadata.csv
Output: ablation_results.csv — resultados completos
        embeddings_h3_optimal.csv — embeddings del mejor modelo
        clusters_h3_optimal.csv — clusters finales
        modelo_optimo.pth — pesos del modelo seleccionado

In [39]:
# Funcion para setear una seed para reproducibilidad
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# ============================================================================
# PASO 0: Cargar y preparar datos
# ============================================================================
firmas = pd.read_csv("../data/auxiliar/firmas_h3.csv", index_col='h3_id')
metadata = pd.read_csv("../data/auxiliar/h3_metadata.csv", index_col='h3_id')

scaler = StandardScaler()
X_std = scaler.fit_transform(firmas)
X_tensor = torch.FloatTensor(X_std).to(device)

h3_ids = firmas.index.tolist()
h3_to_idx = {h: i for i, h in enumerate(h3_ids)}
INPUT_DIM = X_std.shape[1]

print(f"Hexágonos: {len(firmas)}, Dimensiones: {INPUT_DIM}")

# ============================================================================
# PASO 1: Construir grafo de vecindad H3
# ============================================================================

edges_src, edges_dst = [], []
for h3_id in h3_ids:
    for vecino in h3.grid_disk(h3_id, 1):
        if vecino != h3_id and vecino in h3_to_idx:
            edges_src.append(h3_to_idx[h3_id])
            edges_dst.append(h3_to_idx[vecino])

# Extender a k-ring=2 para conectividad
for h3_id in h3_ids:
    for vecino in h3.grid_disk(h3_id, 2):
        if vecino != h3_id and vecino in h3_to_idx:
            edges_src.append(h3_to_idx[h3_id])
            edges_dst.append(h3_to_idx[vecino])

# Deduplicar
edge_set = set(zip(edges_src, edges_dst))
edges_src, edges_dst = zip(*edge_set)
edges_src, edges_dst = list(edges_src), list(edges_dst)

edge_index_full = torch.tensor([edges_src, edges_dst], dtype=torch.long).to(device)
print(f"Grafo: {len(h3_ids)} nodos, {len(edges_src)} aristas")

Hexágonos: 1061, Dimensiones: 45
Grafo: 1061 nodos, 16390 aristas


In [40]:

# ============================================================================
# PASO 2: Train/Val split de aristas (para link prediction)
# ============================================================================
n_edges = edge_index_full.shape[1]
perm = torch.randperm(n_edges)
n_val = int(0.1 * n_edges)  # 10% para validación

val_edge_idx = perm[:n_val]
train_edge_idx = perm[n_val:]

edge_index_train = edge_index_full[:, train_edge_idx]
edge_index_val = edge_index_full[:, val_edge_idx]

print(f"  Aristas entrenamiento: {edge_index_train.shape[1]}")
print(f"  Aristas validación: {edge_index_val.shape[1]}")

  Aristas entrenamiento: 14751
  Aristas validación: 1639


In [41]:
# ============================================================================
# PASO 3: Definir arquitecturas
# ============================================================================

class FlexibleGraphEncoder(nn.Module):
    """Encoder GCN con profundidad y dimensiones configurables."""
    def __init__(self, in_dim, hidden_dim, out_dim, n_layers, dropout=0.1):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        self.dropout = nn.Dropout(dropout)
        self.n_layers = n_layers
        
        # Primera capa
        self.convs.append(GCNConv(in_dim, hidden_dim))
        self.bns.append(nn.BatchNorm1d(hidden_dim))
        
        # Capas intermedias
        for _ in range(n_layers - 2):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
        
        # Última capa (salida)
        self.convs.append(GCNConv(hidden_dim, out_dim))
    
    def forward(self, x, edge_index):
        for i in range(self.n_layers - 1):
            x = self.convs[i](x, edge_index)
            x = self.bns[i](x)
            x = torch.relu(x)
            x = self.dropout(x)
        x = self.convs[-1](x, edge_index)
        return x

class FlexibleVGAEEncoder(nn.Module):
    """Encoder variacional con profundidad y dimensiones configurables."""
    def __init__(self, in_dim, hidden_dim, out_dim, n_layers, dropout=0.1):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        self.dropout = nn.Dropout(dropout)
        self.n_layers = n_layers
        
        # Primera capa
        self.convs.append(GCNConv(in_dim, hidden_dim))
        self.bns.append(nn.BatchNorm1d(hidden_dim))
        
        # Capas intermedias
        for _ in range(n_layers - 2):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
        
        # Dos cabezas
        self.conv_mu = GCNConv(hidden_dim, out_dim)
        self.conv_logvar = GCNConv(hidden_dim, out_dim)
    
    def forward(self, x, edge_index):
        for i in range(self.n_layers - 1):
            x = self.convs[i](x, edge_index)
            x = self.bns[i](x)
            x = torch.relu(x)
            x = self.dropout(x)
        mu = self.conv_mu(x, edge_index)
        logvar = self.conv_logvar(x, edge_index)
        return mu, logvar

class GAEModel(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
    
    def encode(self, x, edge_index):
        return self.encoder(x, edge_index)
    
    def decode(self, z, edge_idx):
        src, dst = edge_idx
        return (z[src] * z[dst]).sum(dim=1)
    
    def recon_loss(self, z, pos_edges, neg_edges):
        pos_score = self.decode(z, pos_edges)
        neg_score = self.decode(z, neg_edges)
        pos_loss = -torch.log(torch.sigmoid(pos_score) + 1e-8).mean()
        neg_loss = -torch.log(1 - torch.sigmoid(neg_score) + 1e-8).mean()
        return pos_loss + neg_loss

class VGAEModel(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
    
    def encode(self, x, edge_index):
        return self.encoder(x, edge_index)
    
    def reparametrize(self, mu, logvar):
        if self.training:
            return mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
        return mu
    
    def decode(self, z, edge_idx):
        src, dst = edge_idx
        return (z[src] * z[dst]).sum(dim=1)
    
    def recon_loss(self, z, pos_edges, neg_edges):
        pos_score = self.decode(z, pos_edges)
        neg_score = self.decode(z, neg_edges)
        pos_loss = -torch.log(torch.sigmoid(pos_score) + 1e-8).mean()
        neg_loss = -torch.log(1 - torch.sigmoid(neg_score) + 1e-8).mean()
        return pos_loss + neg_loss
    
    def kl_loss(self, mu, logvar):
        return -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

# ============================================================================
# PASO 4: Funciones de evaluación
# ============================================================================

def evaluate_link_prediction(model, z, val_edges, num_nodes):
    """AUC y AP sobre aristas de validación."""
    model.eval()
    with torch.no_grad():
        # Positivas
        pos_scores = model.decode(z, val_edges).cpu().numpy()
        
        # Negativas (misma cantidad)
        neg_edges = negative_sampling(
            edge_index=edge_index_full,
            num_nodes=num_nodes,
            num_neg_samples=val_edges.size(1)
        )
        neg_scores = model.decode(z, neg_edges).cpu().numpy()
        
        # Métricas
        labels = np.concatenate([np.ones(len(pos_scores)), np.zeros(len(neg_scores))])
        scores = np.concatenate([pos_scores, neg_scores])
        
        # Aplicar sigmoid
        scores = 1 / (1 + np.exp(-scores))
        
        auc = roc_auc_score(labels, scores)
        ap = average_precision_score(labels, scores)
    return auc, ap

def evaluate_clustering(embeddings, k_range=range(3, 16)):
    """Encuentra mejor K y retorna métricas."""
    best_sil, best_k = -1, 3
    for k in k_range:
        km = KMeans(n_clusters=k, n_init=10, random_state=42)
        labels = km.fit_predict(embeddings)
        sil = silhouette_score(embeddings, labels)
        if sil > best_sil:
            best_sil = sil
            best_k = k
    km = KMeans(n_clusters=best_k, n_init=20, random_state=42)
    labels = km.fit_predict(embeddings)
    ch = calinski_harabasz_score(embeddings, labels)
    return best_k, best_sil, ch, labels

def train_model(model_type, hidden_dim, emb_dim, n_layers, seed, epochs=300, patience=50):
    """Entrena un modelo y retorna embeddings + métricas."""
    set_seed(seed)
    
    if model_type == 'GAE':
        encoder = FlexibleGraphEncoder(INPUT_DIM, hidden_dim, emb_dim, n_layers).to(device)
        model = GAEModel(encoder).to(device)
    else:
        encoder = FlexibleVGAEEncoder(INPUT_DIM, hidden_dim, emb_dim, n_layers).to(device)
        model = VGAEModel(encoder).to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=15, factor=0.5)
    
    best_loss = float('inf')
    patience_counter = 0
    
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        
        neg_edges = negative_sampling(
            edge_index=edge_index_train,
            num_nodes=len(h3_ids),
            num_neg_samples=edge_index_train.size(1)
        )
        
        if model_type == 'GAE':
            z = model.encode(X_tensor, edge_index_train)
            loss = model.recon_loss(z, edge_index_train, neg_edges)
        else:
            mu, logvar = model.encode(X_tensor, edge_index_train)
            z = model.reparametrize(mu, logvar)
            loss = model.recon_loss(z, edge_index_train, neg_edges) + 0.001 * model.kl_loss(mu, logvar)
        
        loss.backward()
        optimizer.step()
        scheduler.step(loss.item())
        
        if loss.item() < best_loss:
            best_loss = loss.item()
            patience_counter = 0
            best_state = model.state_dict().copy()
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            break
    
    # Cargar mejor modelo y extraer embeddings
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        if model_type == 'GAE':
            z = model.encode(X_tensor, edge_index_full)
        else:
            mu, logvar = model.encode(X_tensor, edge_index_full)
            z = mu  # Usar media como embedding
    
    embeddings = z.cpu().numpy()
    
    # Link prediction sobre validación
    auc, ap = evaluate_link_prediction(model, z, edge_index_val, len(h3_ids))
    
    # Clustering
    best_k, sil, ch, labels = evaluate_clustering(embeddings)
    
    return {
        'embeddings': embeddings,
        'labels': labels,
        'model_state': best_state,
        'train_loss': best_loss,
        'val_auc': auc,
        'val_ap': ap,
        'best_k': best_k,
        'silhouette': sil,
        'calinski_harabasz': ch,
        'model': model,
    }


In [47]:

# ============================================================================
# PASO 5: ABLATION STUDY
# ============================================================================
print(f"\n{'='*80}")
print(f"Ablation study: Variación de arquitectura y configuración")
print(f"{'='*80}")

# Configuraciones a probar
model_types = ['GAE', 'VGAE']
hidden_dims = [32, 48]
emb_dims = [8, 12, 16]
n_layers_list = [2, 3]
seeds = [42, 123, 456, 789, 1024]  # 5 seeds para estabilidad

configs = list(itertools.product(model_types, hidden_dims, emb_dims, n_layers_list))
print(f"Configuraciones: {len(configs)} × {len(seeds)} seeds = {len(configs)*len(seeds)} runs")
print(f"Estimación: ~{len(configs)*len(seeds)*0.1:.0f} minutos\n")

all_results = []
total_runs = len(configs) * len(seeds)
run_count = 0

for model_type, hidden_dim, emb_dim, n_layers in configs:
    config_results = []
    
    for seed in seeds:
        run_count += 1
        result = train_model(model_type, hidden_dim, emb_dim, n_layers, seed)
        
        config_results.append({
            'model': model_type,
            'hidden_dim': hidden_dim,
            'emb_dim': emb_dim,
            'n_layers': n_layers,
            'seed': seed,
            'train_loss': result['train_loss'],
            'val_auc': result['val_auc'],
            'val_ap': result['val_ap'],
            'best_k': result['best_k'],
            'silhouette': result['silhouette'],
            'calinski_harabasz': result['calinski_harabasz'],
        })
        all_results.append(config_results[-1])
    
    # Promedios de esta configuración
    avg_sil = np.mean([r['silhouette'] for r in config_results])
    std_sil = np.std([r['silhouette'] for r in config_results])
    avg_auc = np.mean([r['val_auc'] for r in config_results])
    std_auc = np.std([r['val_auc'] for r in config_results])
    
    print(f"  [{run_count:3d}/{total_runs}] {model_type:4s} h={hidden_dim:2d} emb={emb_dim:2d} L={n_layers} -> "
          f"Sil={avg_sil:.3f}±{std_sil:.3f}  AUC={avg_auc:.3f}±{std_auc:.3f}")


Ablation study: Variación de arquitectura y configuración
Configuraciones: 24 × 5 seeds = 120 runs
Estimación: ~12 minutos

  [  5/120] GAE  h=32 emb= 8 L=2 → Sil=0.315±0.010  AUC=0.988±0.002
  [ 10/120] GAE  h=32 emb= 8 L=3 → Sil=0.346±0.014  AUC=0.989±0.002
  [ 15/120] GAE  h=32 emb=12 L=2 → Sil=0.289±0.011  AUC=0.994±0.001
  [ 20/120] GAE  h=32 emb=12 L=3 → Sil=0.315±0.011  AUC=0.993±0.001
  [ 25/120] GAE  h=32 emb=16 L=2 → Sil=0.268±0.009  AUC=0.995±0.001
  [ 30/120] GAE  h=32 emb=16 L=3 → Sil=0.294±0.011  AUC=0.995±0.002
  [ 35/120] GAE  h=48 emb= 8 L=2 → Sil=0.300±0.018  AUC=0.990±0.001
  [ 40/120] GAE  h=48 emb= 8 L=3 → Sil=0.351±0.010  AUC=0.989±0.002
  [ 45/120] GAE  h=48 emb=12 L=2 → Sil=0.287±0.016  AUC=0.992±0.001
  [ 50/120] GAE  h=48 emb=12 L=3 → Sil=0.318±0.011  AUC=0.994±0.001
  [ 55/120] GAE  h=48 emb=16 L=2 → Sil=0.265±0.012  AUC=0.994±0.002
  [ 60/120] GAE  h=48 emb=16 L=3 → Sil=0.298±0.012  AUC=0.996±0.001
  [ 65/120] VGAE h=32 emb= 8 L=2 → Sil=0.313±0.007  AUC=0.9

# TERMINA ARA PT2

# EMPIEZA LUISEN

In [48]:
# ============================================================================
# PASO 6: Análisis de resultados
# ============================================================================
print(f"\n{'='*80}")
print(f"Resultados del ablation study")
print(f"{'='*80}")

df_results = pd.DataFrame(all_results)

# Promediar por configuración
df_avg = df_results.groupby(['model', 'hidden_dim', 'emb_dim', 'n_layers']).agg({
    'train_loss': ['mean', 'std'],
    'val_auc': ['mean', 'std'],
    'val_ap': ['mean', 'std'],
    'silhouette': ['mean', 'std'],
    'calinski_harabasz': ['mean', 'std'],
    'best_k': 'median',
}).reset_index()

# Aplanar columnas
df_avg.columns = ['_'.join(col).strip('_') for col in df_avg.columns]

# Ranking compuesto: normalizar métricas y promediar
from sklearn.preprocessing import MinMaxScaler

metrics_for_ranking = ['val_auc_mean', 'val_ap_mean', 'silhouette_mean', 'calinski_harabasz_mean']
scaler_rank = MinMaxScaler()
df_avg_normalized = pd.DataFrame(
    scaler_rank.fit_transform(df_avg[metrics_for_ranking]),
    columns=metrics_for_ranking
)

# Score compuesto (ponderado: AUC y Silhouette pesan más)
df_avg['composite_score'] = (
    0.30 * df_avg_normalized['val_auc_mean'] +
    0.25 * df_avg_normalized['val_ap_mean'] +
    0.30 * df_avg_normalized['silhouette_mean'] +
    0.15 * df_avg_normalized['calinski_harabasz_mean']
)

# Penalizar varianza alta (inestabilidad)
df_avg['stability_penalty'] = (
    df_avg['silhouette_std'] / (df_avg['silhouette_mean'] + 1e-8) +
    df_avg['val_auc_std'] / (df_avg['val_auc_mean'] + 1e-8)
) / 2

df_avg['final_score'] = df_avg['composite_score'] - 0.1 * df_avg['stability_penalty']

# Top 10
df_avg_sorted = df_avg.sort_values('final_score', ascending=False)

print(f"\n  {'Rank':>4s}  {'Model':>5s}  {'H':>3s}  {'Emb':>3s}  {'L':>1s}  "
      f"{'AUC':>7s}  {'AP':>7s}  {'Sil':>7s}  {'CH':>8s}  {'Score':>6s}")
print(f"  {'-'*65}")

for i, (_, row) in enumerate(df_avg_sorted.head(10).iterrows()):
    print(f"  {i+1:4d}  {row['model']:>5s}  {int(row['hidden_dim']):>3d}  {int(row['emb_dim']):>3d}  "
          f"{int(row['n_layers']):>1d}  {row['val_auc_mean']:>5.3f}  {row['val_ap_mean']:>5.3f}  "
          f"{row['silhouette_mean']:>5.3f}  {row['calinski_harabasz_mean']:>6.1f}  {row['final_score']:>5.3f}")


Resultados del ablation study

  Rank  Model    H  Emb  L      AUC       AP      Sil        CH   Score
  -----------------------------------------------------------------
     1   VGAE   48   12  3  0.994  0.993  0.326   159.8  0.709
     2    GAE   48   12  3  0.994  0.992  0.318   151.2  0.649
     3    GAE   48   16  3  0.996  0.994  0.298   115.3  0.643
     4   VGAE   48    8  3  0.991  0.989  0.349   240.9  0.639
     5   VGAE   32   12  3  0.993  0.991  0.327   166.4  0.632
     6   VGAE   32   16  3  0.995  0.994  0.296   129.0  0.628
     7   VGAE   48   16  3  0.995  0.993  0.295   117.4  0.575
     8    GAE   32   16  3  0.995  0.993  0.294   119.5  0.575
     9   VGAE   48   16  2  0.996  0.995  0.263   105.4  0.553
    10   VGAE   48   12  2  0.995  0.993  0.280   130.7  0.538


In [49]:
# ============================================================================
# PASO 7: ENTRENAR MODELO OPTIMO (con todas las aristas, seed=42)
# ============================================================================

best_config = df_avg_sorted.iloc[0]
opt_model = best_config['model']
opt_hidden = int(best_config['hidden_dim'])
opt_emb = int(best_config['emb_dim'])
opt_layers = int(best_config['n_layers'])

print(f"  Configuración: {opt_model}, hidden={opt_hidden}, emb={opt_emb}, layers={opt_layers}")

# Re-entrenar con TODAS las aristas (no split) para embedding final
set_seed(42) # NOTE la seed puede ser una constante en la configuración

if opt_model == 'GAE':
    encoder_opt = FlexibleGraphEncoder(INPUT_DIM, opt_hidden, opt_emb, opt_layers).to(device)
    model_opt = GAEModel(encoder_opt).to(device)
else:
    encoder_opt = FlexibleVGAEEncoder(INPUT_DIM, opt_hidden, opt_emb, opt_layers).to(device)
    model_opt = VGAEModel(encoder_opt).to(device)

optimizer_opt = torch.optim.Adam(model_opt.parameters(), lr=0.005, weight_decay=1e-5)
scheduler_opt = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_opt, patience=15, factor=0.5)

best_loss_opt = float('inf')
patience_opt = 0
losses_opt = []

for epoch in range(500):  # Más epochs para modelo final
    model_opt.train()
    optimizer_opt.zero_grad()
    
    neg_edges = negative_sampling(
        edge_index=edge_index_full,
        num_nodes=len(h3_ids),
        num_neg_samples=edge_index_full.size(1)
    )
    
    if opt_model == 'GAE':
        z = model_opt.encode(X_tensor, edge_index_full)
        loss = model_opt.recon_loss(z, edge_index_full, neg_edges)
    else:
        mu, logvar = model_opt.encode(X_tensor, edge_index_full)
        z = model_opt.reparametrize(mu, logvar)
        loss = model_opt.recon_loss(z, edge_index_full, neg_edges) + 0.001 * model_opt.kl_loss(mu, logvar)
    
    loss.backward()
    optimizer_opt.step()
    scheduler_opt.step(loss.item())
    losses_opt.append(loss.item())
    
    if loss.item() < best_loss_opt:
        best_loss_opt = loss.item()
        patience_opt = 0
        best_state_opt = model_opt.state_dict().copy()
    else:
        patience_opt += 1
    
    if patience_opt >= 60:
        print(f"  Early stopping epoch {epoch+1}")
        break
    
    if (epoch+1) % 100 == 0:
        print(f"    Epoch {epoch+1}: loss={loss.item():.4f}")

model_opt.load_state_dict(best_state_opt)
model_opt.eval()
with torch.no_grad():
    if opt_model == 'GAE':
        z_opt = model_opt.encode(X_tensor, edge_index_full)
    else:
        mu_opt, _ = model_opt.encode(X_tensor, edge_index_full)
        z_opt = mu_opt

X_optimal = z_opt.cpu().numpy()
print(f"  Mejor loss: {best_loss_opt:.4f}")
print(f"  Embeddings óptimos: {X_optimal.shape}")

  Configuración: VGAE, hidden=48, emb=12, layers=3
    Epoch 100: loss=0.8447
    Epoch 200: loss=0.8241
    Epoch 300: loss=0.8195
  Early stopping epoch 304
  Mejor loss: 0.8105
  Embeddings óptimos: (1061, 12)


In [ ]:
# ============================================================================
# PASO 8: CLUSTERING FINAL + HDBSCAN SOBRE EMBEDDINGS OPTIMOS
# ============================================================================
# TODO aqui capaz podemos modificar el hdbscan con parametros menores a ver si encuentra igual un cluster con configuraciones minimas

# K-Means
best_k_opt, best_sil_opt, best_ch_opt, clusters_opt = evaluate_clustering(X_optimal)
print(f"  K-Means: K={best_k_opt}, Silhouette={best_sil_opt:.3f}, CH={best_ch_opt:.1f}")

# HDBSCAN sobre embeddings óptimos (ahora en dimensionalidad baja)
    
# Probar varias configuraciones de HDBSCAN
print(f"\n  HDBSCAN (sobre {opt_emb}d embeddings):")
best_hdb_sil = -1
best_hdb_config = None

for min_cluster in [10, 15, 20, 25, 30]:
    for min_samples in [3, 5, 7]:
        hdb = hdbscan.HDBSCAN(
            min_cluster_size=min_cluster,
            min_samples=min_samples,
            metric='euclidean',
            cluster_selection_method='eom'
        )
        labels_hdb = hdb.fit_predict(X_optimal)
        n_cl = len(set(labels_hdb)) - (1 if -1 in labels_hdb else 0)
        n_noise = (labels_hdb == -1).sum()
        
        if n_cl > 1:
            mask = labels_hdb != -1
            sil = silhouette_score(X_optimal[mask], labels_hdb[mask])
            if sil > best_hdb_sil and n_noise < len(labels_hdb) * 0.3:  # Max 30% ruido
                best_hdb_sil = sil
                best_hdb_config = (min_cluster, min_samples)
                best_hdb_labels = labels_hdb.copy()
                print(f"    min_cl={min_cluster:2d} min_s={min_samples} → "
                        f"{n_cl} clusters, {n_noise} ruido ({n_noise/len(labels_hdb)*100:.0f}%), Sil={sil:.3f} ✓")

if best_hdb_config:
    clusters_hdb_opt = best_hdb_labels
    print(f"\n  Mejor HDBSCAN: min_cluster={best_hdb_config[0]}, min_samples={best_hdb_config[1]}")
    print(f"  Silhouette: {best_hdb_sil:.3f}")
else:
    print(f"  HDBSCAN no encontró clusters válidos.")
    clusters_hdb_opt = np.full(len(X_optimal), -1)

# GMM
from sklearn.mixture import GaussianMixture
best_bic = float('inf')
for k in range(3, 16):
    gmm = GaussianMixture(n_components=k, covariance_type='full', random_state=42, n_init=3)
    gmm.fit(X_optimal)
    bic = gmm.bic(X_optimal)
    if bic < best_bic:
        best_bic = bic
        best_k_gmm = k
gmm_opt = GaussianMixture(n_components=best_k_gmm, covariance_type='full', random_state=42, n_init=5)
clusters_gmm_opt = gmm_opt.fit_predict(X_optimal)
sil_gmm = silhouette_score(X_optimal, clusters_gmm_opt)
print(f"\n  GMM: K={best_k_gmm}, Silhouette={sil_gmm:.3f}")

  K-Means: K=15, Silhouette=0.331, CH=171.8

  HDBSCAN (sobre 12d embeddings):
  HDBSCAN no encontró clusters válidos.

  GMM: K=15, Silhouette=0.323


In [51]:
# ============================================================================
# PASO 9: Perfiles de clusters (modelo óptimo + K-Means)
# ============================================================================
print(f"\n{'='*80}")
print(f"Perfiles de clusters (Modelo óptimo + K-Means, K={best_k_opt})")
print(f"{'='*80}")

firmas_temp = firmas.copy()
firmas_temp['cluster'] = clusters_opt
firmas_con_meta = firmas_temp.join(metadata)

for c in sorted(firmas_temp['cluster'].unique()):
    grupo = firmas_con_meta[firmas_con_meta['cluster'] == c]
    n = len(grupo)
    print(f"\n▸ Cluster {c} — {n} hexágonos ({n/len(firmas)*100:.1f}%)")
    
    alcs = grupo['alcaldia_dominante'].value_counts().head(3)
    print(f"  Alcaldías: {', '.join([f'{a} ({ct})' for a, ct in alcs.items()])}")
    
    delito_cols = [col for col in firmas.columns if col.startswith('delito_')]
    top_delitos = grupo[delito_cols].mean().sort_values(ascending=False).head(5)
    print(f"  Perfil delictivo:")
    for col, val in top_delitos.items():
        nombre = col.replace('delito_', '').replace('_', ' ').upper()
        print(f"    {nombre:40s} {val*100:5.1f}%")
    
    print(f"  Intensidad: {grupo['intensidad_log'].mean():.2f} (~{np.expm1(grupo['intensidad_log'].mean()):.0f} registros)")
    print(f"  Ratio violencia: {grupo['ratio_violencia'].mean():.3f}")


Perfiles de clusters (Modelo óptimo + K-Means, K=15)

▸ Cluster 0 — 67 hexágonos (6.3%)
  Alcaldías: MIGUEL HIDALGO (27), BENITO JUAREZ (23), ALVARO OBREGON (9)
  Perfil delictivo:
    ROBO SIN VIOLENCIA                        36.4%
    FRAUDE Y DELITOS PATRIMONIALES            19.5%
    ROBO CON VIOLENCIA                         9.9%
    VIOLENCIA FAMILIAR                         6.0%
    DANO EN PROPIEDAD                          5.2%
  Intensidad: 8.04 (~3096 registros)
  Ratio violencia: 0.119

▸ Cluster 1 — 99 hexágonos (9.3%)
  Alcaldías: GUSTAVO A. MADERO (91), VENUSTIANO CARRANZA (8)
  Perfil delictivo:
    ROBO SIN VIOLENCIA                        23.8%
    VIOLENCIA FAMILIAR                        17.2%
    ROBO CON VIOLENCIA                        12.7%
    FRAUDE Y DELITOS PATRIMONIALES            12.3%
    AMENAZAS                                   7.5%
  Intensidad: 7.17 (~1293 registros)
  Ratio violencia: 0.165

▸ Cluster 2 — 65 hexágonos (6.1%)
  Alcaldías: IZTAPALAPA

In [ ]:
# ============================================================================
# PASO 10: EXPORTAR RESULTADOS
# ============================================================================

# Ablation results
df_results.to_csv('../data/results/ablation_results.csv', index=False)
print(f"../data/results/ablation_results.csv ({len(df_results)} runs)")

# Ablation summary
df_avg_sorted.to_csv('../data/results/ablation_summary.csv', index=False)
print(f"../data/results/ablation_summary.csv ({len(df_avg_sorted)} configuraciones)")

# Embeddings óptimos
emb_opt_df = pd.DataFrame(
    X_optimal,
    index=firmas.index,
    columns=[f'emb_{i}' for i in range(X_optimal.shape[1])]
)
emb_opt_df.index.name = 'h3_id'
emb_opt_df.to_csv('../data/results/embeddings_h3_optimal.csv', encoding='utf-8-sig')
print(f"../data/results/embeddings_h3_optimal.csv ({len(emb_opt_df)} × {emb_opt_df.shape[1]})")

# Clusters
clusters_final_df = pd.DataFrame({
    'cluster_kmeans': clusters_opt,
    'cluster_gmm': clusters_gmm_opt,
    'cluster_hdbscan': clusters_hdb_opt,
}, index=firmas.index)
clusters_final_df.index.name = 'h3_id'
clusters_final_df.to_csv('../data/results/clusters_h3_optimal.csv', encoding='utf-8-sig')
print(f"../data/results/clusters_h3_optimal.csv")

# Modelo
torch.save(best_state_opt, '../models/modelo_optimo.pth')
print(f"modelo_optimo.pth ({opt_model}, h={opt_hidden}, emb={opt_emb}, L={opt_layers})")

# Training loss
pd.DataFrame({'epoch': range(1, len(losses_opt)+1), 'loss': losses_opt}).to_csv(
    '../data/results/training_loss_optimal.csv', index=False)
print(f"../data/results/training_loss_optimal.csv")

# Config del modelo óptimo
config_dict = {
    'model_type': opt_model,
    'hidden_dim': opt_hidden,
    'emb_dim': opt_emb,
    'n_layers': opt_layers,
    'best_k_kmeans': best_k_opt,
    'silhouette_kmeans': best_sil_opt,
    'calinski_harabasz': best_ch_opt,
    'train_loss': best_loss_opt,
}
pd.DataFrame([config_dict]).to_csv('../data/results/modelo_optimo_config.csv', index=False)
print(f"../data/results/modelo_optimo_config.csv")

/data/results/ablation_results.csv (120 runs)
✓ ../data/results/ablation_summary.csv (24 configuraciones)
../data/results/embeddings_h3_optimal.csv (1061 × 12)
../data/results/clusters_h3_optimal.csv
modelo_optimo.pth (VGAE, h=48, emb=12, L=3)
../data/results/training_loss_optimal.csv
../data/results/modelo_optimo_config.csv


# TERMINA LUISEN